In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
#1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
#get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop API'
#get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    return data

#print(get_json(BASE_URL + 'status.json')['service'])
#print(get_json(BASE_URL + 'orders/page-99.json'))

Northern Star Shop API


In [0]:
#2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next линкове и враћа записе са свих страна у једној листи
#Помоћ: urljoin(url, page['next'])
#Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа

def fetch_all(first_url: str) -> list:

    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])   
        if page['next'] is not None:
            url = urljoin(url, page['next']) 
        else:
            url = None    

    return records
# 'trenutna stranica' = url; petlja staje kad 'next' postane None -> url postaje None -> while None: False
#print((fetch_all(BASE_URL + 'orders/page-1.json')))
print(type(fetch_all(BASE_URL + 'orders/page-1.json')))

<class 'list'>


In [0]:
#3. Чување сирових података. Написати функцију save_json(records: list, path: str) -> None која листу речника уписује у JSON фајл без икаквих измена.
#json.dump(records, f, ensure_ascii=False, indent=2)
#save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл data/raw/orders.json


def save_json(records: list, path: str) -> None:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(records, f, ensure_ascii=False, indent=2)

save_json((fetch_all(BASE_URL + 'orders/page-1.json')), os.path.join(RAW_DIR, 'orders.json'))

# ensure_ascii=False - dozvoljava srpska slova (c,s,z) da ostanu prava slova u fajlu, ne \u kodovi


In [0]:
#4. Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити у задатку 5

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    with open(path, 'wb') as f: 
        f.write(response.content) 

    return len(response.content)




In [0]:
#5.2. deo za PRODUCTS/MATCHES
download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))
download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))

8022

In [0]:
#download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))
m = pd.read_parquet(os.path.join(RAW_DIR,'matches.parquet'))
m.dtypes
m.head(2)

,match_id,matchday,kickoff_ts,status,home_team_id,away_team_id,home_goals,away_goals,attendance
0,2627001,1,2026-08-15 13:30:00,finished,21,3,2,1,10150
1,2627002,1,2026-08-15 16:00:00,finished,11,18,0,1,12054


In [0]:
#, parse_dates=['kickoff_ts', 'original_kickoff_ts']
c1 = pd.read_csv(os.path.join(RAW_DIR,'teams.csv'))
c2 = pd.read_csv(os.path.join(RAW_DIR,'products.csv'))

In [0]:
#5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams', 'matches'

#5.1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json) - rešeno
#5.2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције download_file 
#5.3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
#Након тога преименовати колоне
#customer_name у customer
#customer_city у city

def extract(raw_dir: str) -> dict:

    #preuzima sva 4 izvora u raw_dir

    teams = download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv')) #5.2
    products = download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))
    matches = download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))

    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(RAW_DIR, 'orders.json')) #5.1

    #5.3 vraća 4 df:
    # pd.json_normalize "spljoska" ugnjezdene recnike (customer.name -> customer_name) u ravne kolone

    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename(columns={'customer_name': 'customer', 'customer_city': 'city'})

    #5.4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
    #По потреби, pd.read_csv(...) или pd.read_parquet(...)

    teams_df = pd.read_csv(os.path.join(RAW_DIR,'teams.csv'))
    products_df = pd.read_csv(os.path.join(RAW_DIR,'products.csv'))
    matches_df = pd.read_parquet(os.path.join(RAW_DIR,'matches.parquet'))
    

    return  {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df,
    }
    


#5.5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)
#print(extract(BASE_URL))

{'orders':          order_id  order_date  ...           customer        city
0       NSS-00001  2026-08-15  ...  NIKOLA STOJANOVIĆ   Zrenjanin
1      nss-00002   2026-08-15  ...   Jovana Pavlović        Čačak
2       NSS-00003  2026-08-15  ...      Sara Živković     Beograd
3       NSS-00004  2026-08-15  ...      Sara Đorđević   Subotica 
4       NSS-00005  2026-08-15  ...     Stefan Popović         Niš
...           ...         ...  ...                ...         ...
1847    NSS-01836  2026-11-15  ...     MILICA POPOVIĆ   SUBOTICA 
1848    NSS-01837  2026-11-15  ...    marko kovačević       Čačak
1849    NSS-01838  2026-11-15  ...      Marko Nikolić      Čačak 
1850    NSS-01839  2026-11-15  ...    Jelena Živković   Zrenjanin
1851    NSS-01840  2026-11-15  ...     ĐORĐE ĐORĐEVIĆ    Novi Sad

[1852 rows x 8 columns], 'products':      product_id           sku     product_name  ...       category     price active
0          1001  NPM-JER-1001  NPM Home Jersey  ...        jerseys   7490.0

In [0]:
#6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame која чисти каталог производа:

#Ако се понавља исти производ, садржати први ред за сваки product_id
#team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред остаје празан (то су производи лиге, без тима)
#category треба да буде без размака, има мала слова
#price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане само float (у сваком случају) Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
#active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало постаје False

#Output: 105 редова, price је типа float64, active је True за 101 производ
#Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves', 'price': 1490.0, 'active': True, ...

In [0]:
#7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame

#order_id, customer, city, payment без размака;
#order_id велика слова
#customer и city -> title()
#Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()  # da ne menjamo original van funkcije
    for col in ['order_id', 'customer', 'city', 'payment']:
        df[col] = df[col].str.strip()

    df['order_id'] = df['order_id'].str.upper()
    df['customer'] = df['customer'].str.title()
    df['city'] = df['city'].str.title().fillna('Unknown')

    return df



In [0]:
#8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple која враћа (unique, duplicates) по order_id, задржава први
#Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return df[~is_dup], df[is_dup]
# keep='first' - prvo pojavljivanje NIJE duplikat; ~ je NOT (negacija) boolean maske


In [0]:
# 9. validate_order - ista logika kao mini projekat, samo pandas-ovi tipovi
#Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа 'OK' за исправан ред, у супротном разлог:

#Провере радити редом и вратити први пронађени проблем:
#order_id не почиње са NSS- -> 'invalid_order'
#quantity недостаје -> 'missing_quantity'
#quantity није цео број већи од нуле -> 'invalid_quantity'
#unit_price није број већи од нуле -> 'invalid_price'
#product_id не постоји међу производима -> 'unknown_product'

#Помоћ:
#pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
#pd.isna(value) препознаје и None и NaN

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')  # 'abc' -> NaN umesto greske
    price = pd.to_numeric(row['unit_price'], errors='coerce')
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(price) or price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    return 'OK'

In [0]:
#10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame
#Типови:
#quantity -> int
#unit_price -> float
#order_date -> датум (pd.to_datetime)
#Нова колона total (quantity * unit_price)
#Из очишћених производа (products) додати product_name, team_code и category (merge по product_id)
#Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price']).astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']
    names = products[['product_name', 'team_code', 'category']]
    return df.merge(names, on='product_id', how='left')
# merge = SQL JOIN analogija - svaka porudzbina "nasledi" ime/kategoriju proizvoda preko product_id

In [0]:
#11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,rejected):

#valid су трансформисани исправни редови
#rejected су очишћени неисправни редови и дупликати, са колоном reason (дупликати за разлог треба да имају 'duplicate')
#Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    df = clean_orders(df)
    unique, duplicates = remove_duplicate_orders(df)
    duplicates = duplicates.assign(reason='duplicate')
    product_ids = set(products['product_id'])
    reasons = unique.apply(lambda row: validate_order(row, product_ids), axis=1)
    valid = transform_orders(unique[reasons == 'OK'], products)
    rejected = pd.concat([unique[reasons != 'OK'].assign(reason=reasons[reasons != 'OK']), duplicates])
    return valid, rejected